In [3]:
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
import umap

DATA_DIR = Path("Outputs/PacketEmbeddings")
SEED = 42

# ---- load ----
X_parts, labels = [], []
for f in sorted(DATA_DIR.glob("*.npy")):
    arr = np.load(f)
    X_parts.append(arr)
    labels += [f.stem] * len(arr)
X = np.vstack(X_parts).astype(np.float32)
labels = np.array(labels)
cats = np.unique(labels)
print(X.shape, len(cats), "categories")


(2202044, 32) 15 categories


In [4]:

rng = np.random.default_rng(SEED)
N_MAX = 50_000
per_cat = max(1, N_MAX // len(cats))
idx = np.concatenate([rng.choice(np.where(labels == c)[0],
                                 min(per_cat, (labels == c).sum()), replace=False)
                      for c in cats])
X, labels = X[idx], labels[idx]

X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
X = StandardScaler().fit_transform(X)

# ---- reduce ----
N_NEIGHBORS = 300
MIN_DIST = 0.001
reducer = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                    metric="cosine")
Z = reducer.fit_transform(X)
print("UMAP done!")

# ---- interactive plot ----
fig = go.Figure()
for c in cats:
    m = labels == c
    fig.add_trace(go.Scattergl(
        x=Z[m, 0], y=Z[m, 1],
        mode="markers",
        name=str(c),
        marker=dict(size=3, opacity=0.6),
        hovertemplate=f"{c}<extra></extra>",
    ))

fig.update_layout(
    title=f"UMAP (n_neighbors={N_NEIGHBORS}, min_dist={MIN_DIST})",
    width=1000, height=800,
    legend=dict(itemsizing="constant", itemclick="toggle", itemdoubleclick="toggleothers"),
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
               scaleanchor="x", scaleratio=1),
    template="plotly_white",
)

fig.write_html("embedding_projection.html", include_plotlyjs="cdn")
fig.show()

UMAP done!
